# Mount St. Helens station SEP: miniSEED → SQL database → helicorders and station map

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_GITHUB_USERNAME/sep-seismic-pipeline/blob/main/notebooks/sep_seismic_pipeline.ipynb)

This notebook is a small, self-contained data pipeline for three days of continuous seismic data from
station **CC.SEP** (the crater station at Mount St. Helens, run by the USGS Cascades Volcano Observatory).

| Step | What happens |
|---|---|
| 1. Setup | Install ObsPy and Cartopy, set configuration |
| 2. Import the miniSEED files | Find the uploaded `SEP01.mseed`, `SEP02.mseed` and `SEP03.mseed`, verify checksums, read with ObsPy |
| 3. Create the database | SQLite schema in third normal form with primary keys, foreign keys, `UNIQUE`/`CHECK` constraints, a trigger and views |
| 4. Load the waveform data | Every sample goes in as a row, keyed by `(segment_id, sample_index)` |
| 5. Station metadata from IRIS | FDSN station web service: SEP itself, the stations of interest (HOA, SUG) and other stations nearby |
| 6. Validate | Integrity checks, row counts against headers, a round-trip comparison and constraint checks |
| 7. Visualize | One figure with a title, a metadata panel, 24-hour helicorders built **from the database**, and a station map |

**Running it:**
1. Upload `SEP01.mseed`, `SEP02.mseed` and `SEP03.mseed` (from
   [CHIGHT-USGS/SeismicData](https://github.com/CHIGHT-USGS/SeismicData)) through Colab's **Files** panel. The
   session's main folder (`/content`) is fine, or a `data/` folder inside it.
2. *Runtime → Run all*.

Station metadata comes from the [IRIS FDSN station service](http://service.iris.edu/fdsnws/station/1/). A full run takes about 2–4 minutes
on a standard CPU runtime, mostly installing packages and loading about 26 million samples.

## 1. Setup
Colab already has NumPy, pandas, SciPy, Matplotlib and Requests. ObsPy (the miniSEED reader) and
Cartopy (map projections and basemap tiles) are installed here.

In [ ]:
%pip install --quiet "obspy>=1.4.2" "cartopy>=0.22"

In [ ]:
import hashlib
import io
import itertools
import math
import sqlite3
import warnings
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from operator import itemgetter
from pathlib import Path

import matplotlib.patheffects as pe
import matplotlib.pyplot as plt
import numpy as np
import obspy
import pandas as pd
import requests
from matplotlib.gridspec import GridSpec
from matplotlib.lines import Line2D
from scipy import signal

# ---- Configuration ---------------------------------------------------------------------------
# Folders searched for the uploaded miniSEED files, in order (Colab's working folder is /content).
DATA_SEARCH_DIRS = (Path("."), Path("data"), Path("../data"), Path("/content"), Path("/content/data"))
# Expected SHA-256 of each file (as published in CHIGHT-USGS/SeismicData), so a corrupted upload is caught.
MSEED_FILES = {
    "SEP01.mseed": "3c888951976d35affc92eb36c5797c3159fd9f819046190ece376729d1a19f05",
    "SEP02.mseed": "2312e9abb3b0fbb5cd5e7a85ee26846fd8aa89d5b752b0758ed71958a40abfa7",
    "SEP03.mseed": "98b143b6d81b7d0bb851713fb38bcd85cc199c9dd3748aa9574242979de51766",
}
OUTPUT_DIR = Path("outputs")
DB_PATH = OUTPUT_DIR / "seismic.db"

# FDSN station web service (IRIS is now operated by EarthScope; the second URL is a fallback).
FDSN_STATION_URLS = (
    "https://service.iris.edu/fdsnws/station/1/query",
    "https://service.earthscope.org/fdsnws/station/1/query",
)
STATIONS_OF_INTEREST = ("HOA", "SUG")
INTEREST_SEARCH_RADIUS_DEG = 1.0  # station codes are not globally unique, so search only near SEP
NEARBY_RADIUS_DEG = 0.15          # about 16 km: other stations operating during the data window

# Helicorder settings
HELI_LINE_MINUTES = 60    # one trace line per hour, 24 lines per day
HELI_HIGHPASS_HZ = 0.5    # removes long-period drift so the lines don't wander
HELI_P99_HEIGHT = 0.2    # the 99th-percentile amplitude is drawn at 0.2 line spacings
HELI_CLIP = 0.9           # clip amplitudes at 0.9 line spacings, as on a paper drum
LOCAL_TZ_NAME, LOCAL_UTC_OFFSET_H = "PST", -8  # all three days fall after the Nov 5, 2023 DST change

OUTPUT_DIR.mkdir(exist_ok=True)
pd.set_option("display.width", 160)
print(f"ObsPy {obspy.__version__} | NumPy {np.__version__} | pandas {pd.__version__} | SQLite {sqlite3.sqlite_version}")

## 2. Import the miniSEED files
The three files are expected to have been uploaded before the run. Each one is looked for in the session's working
folder and in a `data/` folder, and its checksum is verified before it is read. If a file is missing, the cell stops
and says which one.

In [ ]:
def sha256_of(path: Path, chunk_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def locate_mseed_files() -> dict[str, Path]:
    """Find each uploaded miniSEED file in DATA_SEARCH_DIRS. Stops with instructions if any are missing."""
    found, missing = {}, []
    for name in MSEED_FILES:
        path = next((d / name for d in DATA_SEARCH_DIRS if (d / name).is_file()), None)
        if path is None:
            missing.append(name)
        else:
            found[name] = path.resolve()
    if missing:
        searched = ", ".join(str(d.resolve()) for d in DATA_SEARCH_DIRS)
        raise FileNotFoundError(
            f"Missing {', '.join(missing)}. Upload the files with Colab's Files panel (into /content or "
            f"/content/data), then run this cell again. Folders searched: {searched}")
    return found


def verified(path: Path, expected_sha256: str) -> Path:
    actual = sha256_of(path)
    if actual != expected_sha256:
        raise ValueError(f"{path.name}: SHA-256 {actual[:12]}… does not match the expected {expected_sha256[:12]}…")
    return path


mseed_paths = {name: verified(path, MSEED_FILES[name]) for name, path in locate_mseed_files().items()}
for name, path in mseed_paths.items():
    print(f"{name}: {path}")
streams = {name: obspy.read(str(path), format="MSEED") for name, path in mseed_paths.items()}


def seed_id(stats) -> str:
    return f"{stats.network}.{stats.station}.{stats.location}.{stats.channel}"


def segment_end(stats) -> datetime:
    """Exclusive end time of a trace: the time one sample after its last sample."""
    return stats.starttime.datetime + timedelta(seconds=stats.npts / stats.sampling_rate)


import_summary = pd.DataFrame(
    [
        {
            "file": name,
            "size_MB": round(mseed_paths[name].stat().st_size / 1e6, 2),
            "seed_id": seed_id(st[0].stats),
            "segments": len(st),
            "start_utc": min(tr.stats.starttime.datetime for tr in st),
            "end_utc": max(segment_end(tr.stats) for tr in st),
            "samples": sum(tr.stats.npts for tr in st),
            "sample_rate_hz": st[0].stats.sampling_rate,
            "encoding": st[0].stats.mseed.encoding,
            "record_length": st[0].stats.mseed.record_length,
        }
        for name, st in streams.items()
    ]
)
import_summary

## 3. Create the database

SQLite comes with Python and runs in Colab without a server. The schema is in **third normal form**:
each table describes one thing, every non-key column depends only on its table's key, and nothing that
can be computed is stored.

| Table | One row per | Key / important constraints |
|---|---|---|
| `network` | FDSN network (e.g. `CC`) | `code` UNIQUE, 1–2 upper-case characters |
| `station` | station within a network | FK → network, UNIQUE (network, code) |
| `station_epoch` | period with fixed coordinates (from IRIS) | FK → station, lat/lon range CHECKs, end > start |
| `channel` | location + channel code at a station | FK → station, UNIQUE (station, location, channel) |
| `channel_epoch` | channel response/sensor period (from IRIS) | FK → channel, sample rate > 0, end > start |
| `source_file` | imported miniSEED file | UNIQUE file name and SHA-256, record length a power of two |
| `trace_segment` | contiguous run of samples in a file | FK → channel and file, npts > 0, UNIQUE (channel, start) |
| `sample` | one digitizer count | PK (segment_id, sample_index); a trigger keeps the index below the segment's npts |

Design notes:
* **Times** are stored as `INTEGER` microseconds since 1970-01-01 UTC. That is exact (miniSEED times go to
  1 µs, with blockette 1001), sorts correctly and is easy to work with. The views show them as ISO-8601 text.
* **No derived columns.** A segment's end time is `start + npts / sample_rate`, and a sample's time is
  `start + index / sample_rate`, so both are computed in views (`v_segment`, `v_gap`) rather than stored,
  where they could drift out of step.
* **Waveform samples are rows**, not arrays packed into a BLOB, so the table stays in first normal form and can
  be queried directly in SQL. Using `WITHOUT ROWID` stores each row inside the primary-key B-tree, which keeps the
  table compact (about 120 MB per day of 100 Hz data).
* miniSEED writes an empty location code as `--`; it is stored as `''`, matching the FDSN convention.

In [ ]:
SCHEMA_SQL = """
CREATE TABLE network (
    network_id    INTEGER PRIMARY KEY,
    code          TEXT    NOT NULL UNIQUE
                  CHECK (length(code) BETWEEN 1 AND 2 AND code = upper(code))
);

CREATE TABLE station (
    station_id    INTEGER PRIMARY KEY,
    network_id    INTEGER NOT NULL REFERENCES network(network_id) ON DELETE CASCADE,
    code          TEXT    NOT NULL CHECK (length(code) BETWEEN 1 AND 5 AND code = upper(code)),
    UNIQUE (network_id, code)
);

CREATE TABLE station_epoch (
    station_epoch_id INTEGER PRIMARY KEY,
    station_id       INTEGER NOT NULL REFERENCES station(station_id) ON DELETE CASCADE,
    start_time_us    INTEGER NOT NULL,
    end_time_us      INTEGER,                         -- NULL = still open
    latitude         REAL    NOT NULL CHECK (latitude  BETWEEN  -90 AND  90),
    longitude        REAL    NOT NULL CHECK (longitude BETWEEN -180 AND 180),
    elevation_m      REAL,
    site_name        TEXT,
    UNIQUE (station_id, start_time_us),
    CHECK (end_time_us IS NULL OR end_time_us > start_time_us)
);

CREATE TABLE channel (
    channel_id    INTEGER PRIMARY KEY,
    station_id    INTEGER NOT NULL REFERENCES station(station_id) ON DELETE CASCADE,
    location_code TEXT    NOT NULL DEFAULT '' CHECK (length(location_code) <= 2),
    code          TEXT    NOT NULL CHECK (length(code) = 3 AND code = upper(code)),
    UNIQUE (station_id, location_code, code)
);

CREATE TABLE channel_epoch (
    channel_epoch_id   INTEGER PRIMARY KEY,
    channel_id         INTEGER NOT NULL REFERENCES channel(channel_id) ON DELETE CASCADE,
    start_time_us      INTEGER NOT NULL,
    end_time_us        INTEGER,
    sample_rate_hz     REAL    CHECK (sample_rate_hz > 0),
    depth_m            REAL,
    azimuth_deg        REAL    CHECK (azimuth_deg BETWEEN 0 AND 360),
    dip_deg            REAL    CHECK (dip_deg BETWEEN -90 AND 90),
    sensor_description TEXT,
    scale              REAL,
    scale_freq_hz      REAL,
    scale_units        TEXT,
    UNIQUE (channel_id, start_time_us),
    CHECK (end_time_us IS NULL OR end_time_us > start_time_us)
);

CREATE TABLE source_file (
    file_id       INTEGER PRIMARY KEY,
    file_name     TEXT    NOT NULL UNIQUE,
    sha256        TEXT    NOT NULL UNIQUE CHECK (length(sha256) = 64),
    size_bytes    INTEGER NOT NULL CHECK (size_bytes > 0),
    record_length INTEGER NOT NULL CHECK (record_length IN (128, 256, 512, 1024, 2048, 4096, 8192, 16384)),
    record_count  INTEGER NOT NULL CHECK (record_count > 0),
    encoding      TEXT    NOT NULL,
    byte_order    TEXT    NOT NULL CHECK (byte_order IN ('>', '<')),
    imported_at   TEXT    NOT NULL DEFAULT (strftime('%Y-%m-%dT%H:%M:%SZ', 'now'))
);

CREATE TABLE trace_segment (
    segment_id     INTEGER PRIMARY KEY,
    channel_id     INTEGER NOT NULL REFERENCES channel(channel_id) ON DELETE CASCADE,
    file_id        INTEGER NOT NULL REFERENCES source_file(file_id) ON DELETE CASCADE,
    start_time_us  INTEGER NOT NULL,
    sample_rate_hz REAL    NOT NULL CHECK (sample_rate_hz > 0),
    npts           INTEGER NOT NULL CHECK (npts > 0),
    data_quality   TEXT    NOT NULL CHECK (data_quality IN ('D', 'R', 'Q', 'M')),
    UNIQUE (channel_id, start_time_us)
);
CREATE INDEX idx_trace_segment_file ON trace_segment(file_id);

CREATE TABLE sample (
    segment_id   INTEGER NOT NULL REFERENCES trace_segment(segment_id) ON DELETE CASCADE,
    sample_index INTEGER NOT NULL CHECK (sample_index >= 0),
    counts       INTEGER NOT NULL,
    PRIMARY KEY (segment_id, sample_index)
) WITHOUT ROWID;

-- A sample's index must fall inside its segment (a cross-table rule, so it needs a trigger).
CREATE TRIGGER trg_sample_index_in_segment
BEFORE INSERT ON sample
WHEN NEW.sample_index >= (SELECT npts FROM trace_segment WHERE segment_id = NEW.segment_id)
BEGIN
    SELECT RAISE(ABORT, 'sample_index is outside its trace_segment');
END;

-- ---------- Views: derived values are computed here, not stored ----------
CREATE VIEW v_segment AS
SELECT s.segment_id, s.channel_id, s.file_id, f.file_name,
       n.code || '.' || st.code || '.' || c.location_code || '.' || c.code AS seed_id,
       s.start_time_us,
       s.start_time_us + CAST(ROUND(s.npts * 1e6 / s.sample_rate_hz) AS INTEGER) AS end_time_us,
       strftime('%Y-%m-%dT%H:%M:%fZ', s.start_time_us / 1e6, 'unixepoch') AS start_utc,
       strftime('%Y-%m-%dT%H:%M:%fZ', (s.start_time_us + s.npts * 1e6 / s.sample_rate_hz) / 1e6, 'unixepoch') AS end_utc,
       s.npts, s.sample_rate_hz, s.data_quality
FROM trace_segment s
JOIN channel     c  ON c.channel_id  = s.channel_id
JOIN station     st ON st.station_id = c.station_id
JOIN network     n  ON n.network_id  = st.network_id
JOIN source_file f  ON f.file_id     = s.file_id;

CREATE VIEW v_gap AS
SELECT channel_id, file_name,
       prev_end_us AS gap_start_us, start_time_us AS gap_end_us,
       strftime('%Y-%m-%dT%H:%M:%fZ', prev_end_us / 1e6, 'unixepoch') AS gap_start_utc,
       (start_time_us - prev_end_us) / 1e6 AS gap_seconds
FROM (
    SELECT v.*, LAG(end_time_us) OVER (PARTITION BY channel_id, file_id ORDER BY start_time_us) AS prev_end_us
    FROM v_segment v
)
WHERE prev_end_us IS NOT NULL
  AND start_time_us - prev_end_us > 0.5e6 / sample_rate_hz;

CREATE VIEW v_station_epoch AS
SELECT n.code AS network, s.code AS station, e.*,
       strftime('%Y-%m-%d', e.start_time_us / 1e6, 'unixepoch') AS start_date,
       strftime('%Y-%m-%d', e.end_time_us   / 1e6, 'unixepoch') AS end_date
FROM station_epoch e
JOIN station s ON s.station_id = e.station_id
JOIN network n ON n.network_id = s.network_id;
"""


def create_database(path: Path) -> sqlite3.Connection:
    """Create a fresh database at `path`. Each run starts from scratch, so it can be re-run safely."""
    path.unlink(missing_ok=True)
    conn = sqlite3.connect(path)
    conn.execute("PRAGMA foreign_keys = ON")   # SQLite enforces foreign keys only when asked to
    conn.execute("PRAGMA journal_mode = WAL")
    conn.execute("PRAGMA synchronous = NORMAL")
    conn.executescript(SCHEMA_SQL)
    return conn


conn = create_database(DB_PATH)
pd.read_sql("SELECT type, name FROM sqlite_master WHERE name NOT LIKE 'sqlite_%' ORDER BY type, name", conn)

## 4. Load the waveform data
Each file is loaded in its own transaction, so a failure part-way through leaves no half-loaded file behind.
Samples go in through `executemany` with a generator, so no 8.6-million-row Python list is ever built.

In [ ]:
EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)


def to_us(t) -> int:
    """Microseconds since the Unix epoch, from an ObsPy UTCDateTime or a timezone-aware datetime."""
    if hasattr(t, "ns"):  # ObsPy UTCDateTime keeps exact integer nanoseconds
        return t.ns // 1000
    return (t - EPOCH) // timedelta(microseconds=1)


def from_us(us: int) -> datetime:
    return EPOCH + timedelta(microseconds=int(us))


def normalize_location(loc: str) -> str:
    """miniSEED writes an empty location as '--' or spaces; store it as ''."""
    return "" if loc.strip(" -") == "" else loc.strip()


def ensure_network(conn, code: str) -> int:
    conn.execute("INSERT OR IGNORE INTO network (code) VALUES (?)", (code,))
    return conn.execute("SELECT network_id FROM network WHERE code = ?", (code,)).fetchone()[0]


def ensure_station(conn, network: str, station: str) -> int:
    net_id = ensure_network(conn, network)
    conn.execute("INSERT OR IGNORE INTO station (network_id, code) VALUES (?, ?)", (net_id, station))
    return conn.execute(
        "SELECT station_id FROM station WHERE network_id = ? AND code = ?", (net_id, station)
    ).fetchone()[0]


def ensure_channel(conn, network: str, station: str, location: str, channel: str) -> int:
    sta_id = ensure_station(conn, network, station)
    conn.execute(
        "INSERT OR IGNORE INTO channel (station_id, location_code, code) VALUES (?, ?, ?)",
        (sta_id, location, channel),
    )
    return conn.execute(
        "SELECT channel_id FROM channel WHERE station_id = ? AND location_code = ? AND code = ?",
        (sta_id, location, channel),
    ).fetchone()[0]


def load_mseed_file(conn, path: Path, stream) -> int:
    """Insert one miniSEED file (file row, segments and samples) in a single transaction."""
    mseed = stream[0].stats.mseed
    # File-level attributes must be the same for every trace, or they would not belong on source_file.
    assert all(tr.stats.mseed.encoding == mseed.encoding for tr in stream), "mixed encodings in one file"
    assert all(tr.stats.mseed.record_length == mseed.record_length for tr in stream), "mixed record lengths"
    with conn:
        file_id = conn.execute(
            """INSERT INTO source_file
                   (file_name, sha256, size_bytes, record_length, record_count, encoding, byte_order)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (path.name, sha256_of(path), path.stat().st_size, mseed.record_length,
             mseed.number_of_records, mseed.encoding, mseed.byteorder),
        ).lastrowid
        for tr in sorted(stream, key=lambda t: t.stats.starttime.datetime):
            s = tr.stats
            channel_id = ensure_channel(conn, s.network, s.station, normalize_location(s.location), s.channel)
            segment_id = conn.execute(
                """INSERT INTO trace_segment
                       (channel_id, file_id, start_time_us, sample_rate_hz, npts, data_quality)
                   VALUES (?, ?, ?, ?, ?, ?)""",
                (channel_id, file_id, to_us(s.starttime), s.sampling_rate, s.npts, s.mseed.dataquality),
            ).lastrowid
            conn.executemany(
                "INSERT INTO sample (segment_id, sample_index, counts) VALUES (?, ?, ?)",
                zip(itertools.repeat(segment_id), range(s.npts), tr.data.tolist()),
            )
    return file_id


for name, stream in streams.items():
    t0 = datetime.now()
    file_id = load_mseed_file(conn, mseed_paths[name], stream)
    print(f"{name}: file_id={file_id}, {sum(tr.stats.npts for tr in stream):,} samples "
          f"in {(datetime.now() - t0).total_seconds():.1f} s")

pd.read_sql("SELECT segment_id, file_name, seed_id, start_utc, end_utc, npts FROM v_segment", conn)

## 5. Station metadata from the IRIS FDSN station service
The service's `format=text` output is a pipe-separated table, which pandas reads directly. Three queries:

1. **Channel level** for the channel in the miniSEED files (`CC.SEP.--.EHZ`), limited to the data window:
   sensor, sample rate, sensitivity and coordinates.
2. **Stations of interest**, `HOA` and `SUG`, within 1° of SEP. Station codes are only unique within a network,
   so the search is kept to the area.
3. **Nearby stations** (within about 16 km) that were operating during the data window, as extra map context.

All epochs returned are stored in `station_epoch` and `channel_epoch`.

In [ ]:
TEXT_COLUMN_RENAMES = {
    "sitename": "site_name", "starttime": "start_time", "endtime": "end_time",
    "sensordescription": "sensor_description", "scalefreq": "scale_freq", "scaleunits": "scale_units",
    "samplerate": "sample_rate", "location": "location", "channel": "channel",
}
NUMERIC_COLUMNS = ("latitude", "longitude", "elevation", "depth", "azimuth", "dip", "scale", "scale_freq", "sample_rate")


def fdsn_station_query(level: str, **params) -> pd.DataFrame:
    """Query the FDSN station service (format=text) and return a tidy DataFrame (empty if no matches)."""
    query = {"level": level, "format": "text", **params}
    last_error = None
    for url in FDSN_STATION_URLS:
        try:
            resp = requests.get(url, params=query, timeout=30)
        except requests.RequestException as exc:
            last_error = exc
            continue
        if resp.status_code in (204, 404):  # the service's "no data" responses
            return pd.DataFrame()
        if resp.ok:
            df = pd.read_csv(io.StringIO(resp.text), sep="|", dtype=str, keep_default_na=False)
            df.columns = [c.strip().lstrip("#").strip().lower() for c in df.columns]
            df = df.rename(columns=TEXT_COLUMN_RENAMES).apply(lambda col: col.str.strip())
            for col in NUMERIC_COLUMNS:
                if col in df:
                    df[col] = pd.to_numeric(df[col], errors="coerce")
            return df
        last_error = RuntimeError(f"HTTP {resp.status_code} from {url}: {resp.text[:200]}")
    raise RuntimeError(f"FDSN station query failed: {last_error}")


def parse_fdsn_time_us(value: str) -> int | None:
    """FDSN text times look like 2004-10-08T00:00:00 or …:00.0000. Blank means open-ended."""
    if not value:
        return None
    dt = datetime.fromisoformat(value.rstrip("Z"))
    return to_us(dt.replace(tzinfo=timezone.utc))


# Data window and identity of the stream we imported
seg = pd.read_sql("SELECT MIN(start_time_us) AS t0, MAX(end_time_us) AS t1 FROM v_segment", conn).iloc[0]
window_start, window_end = from_us(seg.t0), from_us(seg.t1)
first_stats = next(iter(streams.values()))[0].stats
NET, STA, CHA = first_stats.network, first_stats.station, first_stats.channel
LOC = normalize_location(first_stats.location)
iso = lambda d: d.strftime("%Y-%m-%dT%H:%M:%S")

# 1. Channel-level metadata for the imported channel
channel_meta = fdsn_station_query(
    "channel", net=NET, sta=STA, loc=LOC or "--", cha=CHA,
    starttime=iso(window_start), endtime=iso(window_end),
)
if channel_meta.empty:
    raise RuntimeError(f"IRIS returned no channel metadata for {NET}.{STA}.{LOC or '--'}.{CHA}")
primary_station_meta = fdsn_station_query("station", net=NET, sta=STA)  # every epoch of SEP
ref_lat, ref_lon = channel_meta.iloc[0][["latitude", "longitude"]]

# 2. Stations of interest, searched near SEP
interest_meta = fdsn_station_query(
    "station", sta=",".join(STATIONS_OF_INTEREST),
    latitude=ref_lat, longitude=ref_lon, maxradius=INTEREST_SEARCH_RADIUS_DEG,
)
missing = set(STATIONS_OF_INTEREST) - set(interest_meta.get("station", []))
if missing:
    warnings.warn(f"No IRIS metadata found near {STA} for: {sorted(missing)}")

# 3. Other stations operating nearby during the data window
nearby_meta = fdsn_station_query(
    "station", latitude=ref_lat, longitude=ref_lon, maxradius=NEARBY_RADIUS_DEG,
    starttime=iso(window_start), endtime=iso(window_end),
)

print(f"Channel epochs: {len(channel_meta)} | {STA} station epochs: {len(primary_station_meta)} | "
      f"stations-of-interest epochs: {len(interest_meta)} | nearby stations: {len(nearby_meta)}")
channel_meta.T

In [ ]:
def load_station_epochs(conn, df: pd.DataFrame) -> int:
    """Store station-level rows (one per epoch). Re-inserting an existing epoch is a no-op."""
    rows = 0
    with conn:
        for r in df.itertuples(index=False):
            station_id = ensure_station(conn, r.network, r.station)
            cur = conn.execute(
                """INSERT OR IGNORE INTO station_epoch
                       (station_id, start_time_us, end_time_us, latitude, longitude, elevation_m, site_name)
                   VALUES (?, ?, ?, ?, ?, ?, ?)""",
                (station_id, parse_fdsn_time_us(r.start_time), parse_fdsn_time_us(r.end_time),
                 r.latitude, r.longitude, r.elevation, r.site_name),
            )
            rows += cur.rowcount
    return rows


def load_channel_epochs(conn, df: pd.DataFrame) -> int:
    rows = 0
    with conn:
        for r in df.itertuples(index=False):
            channel_id = ensure_channel(conn, r.network, r.station, normalize_location(r.location), r.channel)
            cur = conn.execute(
                """INSERT OR IGNORE INTO channel_epoch
                       (channel_id, start_time_us, end_time_us, sample_rate_hz, depth_m, azimuth_deg, dip_deg,
                        sensor_description, scale, scale_freq_hz, scale_units)
                   VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)""",
                (channel_id, parse_fdsn_time_us(r.start_time), parse_fdsn_time_us(r.end_time),
                 r.sample_rate, r.depth, r.azimuth, r.dip, r.sensor_description or None,
                 r.scale, r.scale_freq, r.scale_units or None),
            )
            rows += cur.rowcount
    return rows


n_sta = sum(load_station_epochs(conn, df) for df in (primary_station_meta, interest_meta, nearby_meta) if not df.empty)
n_cha = load_channel_epochs(conn, channel_meta)
print(f"Inserted {n_sta} station epochs and {n_cha} channel epochs")
pd.read_sql(
    "SELECT network, station, site_name, latitude, longitude, elevation_m, start_date, end_date "
    "FROM v_station_epoch ORDER BY network, station, start_time_us", conn
)

## 6. Validate the database
* SQLite's own `integrity_check` and `foreign_key_check`
* every segment has exactly `npts` sample rows
* a round trip: samples read back from SQL match the samples ObsPy decoded, exactly
* the constraints reject bad rows

In [ ]:
assert conn.execute("PRAGMA integrity_check").fetchone()[0] == "ok"
assert conn.execute("PRAGMA foreign_key_check").fetchall() == []

counts = pd.read_sql(
    """SELECT s.segment_id, s.npts, COUNT(x.sample_index) AS sample_rows
       FROM trace_segment s LEFT JOIN sample x ON x.segment_id = s.segment_id
       GROUP BY s.segment_id""", conn)
assert (counts.npts == counts.sample_rows).all(), counts[counts.npts != counts.sample_rows]


def fetch_segment_counts(conn, segment_id: int, npts: int) -> np.ndarray:
    cur = conn.execute("SELECT counts FROM sample WHERE segment_id = ? ORDER BY sample_index", (segment_id,))
    return np.fromiter(map(itemgetter(0), cur), dtype=np.int64, count=npts)


# Round trip for every segment of the first file
first_name, first_stream = next(iter(streams.items()))
db_segments = conn.execute(
    "SELECT segment_id, npts FROM v_segment WHERE file_name = ? ORDER BY start_time_us", (first_name,)
).fetchall()
for (segment_id, npts), tr in zip(db_segments, sorted(first_stream, key=lambda t: t.stats.starttime.datetime)):
    assert np.array_equal(fetch_segment_counts(conn, segment_id, npts), tr.data), f"segment {segment_id} differs"


def expect_rejected(sql: str, params: tuple) -> str:
    """Run a statement that must fail; roll it back and return the database's error message."""
    try:
        with conn:
            conn.execute(sql, params)
    except sqlite3.DatabaseError as exc:
        return f"{type(exc).__name__}: {exc}"
    raise AssertionError(f"constraint did not fire: {sql}")


some_station = conn.execute("SELECT station_id FROM station LIMIT 1").fetchone()[0]
constraint_checks = pd.DataFrame(
    [
        ("Duplicate network code", expect_rejected("INSERT INTO network (code) VALUES (?)", (NET,))),
        ("Lower-case network code", expect_rejected("INSERT INTO network (code) VALUES (?)", ("xx",))),
        ("Latitude out of range", expect_rejected(
            "INSERT INTO station_epoch (station_id, start_time_us, latitude, longitude) VALUES (?, 0, 123, 0)",
            (some_station,))),
        ("Epoch ends before it starts", expect_rejected(
            "INSERT INTO station_epoch (station_id, start_time_us, end_time_us, latitude, longitude) "
            "VALUES (?, 10, 5, 0, 0)", (some_station,))),
        ("Sample for a missing segment (FK)", expect_rejected(
            "INSERT INTO sample VALUES (?, 0, 0)", (999_999,))),
        ("Sample index beyond npts (trigger)", expect_rejected(
            "INSERT INTO sample VALUES (?, ?, 0)", (db_segments[0][0], db_segments[0][1]))),
        ("Zero-length segment", expect_rejected(
            "INSERT INTO trace_segment (channel_id, file_id, start_time_us, sample_rate_hz, npts, data_quality) "
            "VALUES (1, 1, 0, 100, 0, 'D')", ())),
    ],
    columns=["check", "database response"],
)
table_rows = pd.read_sql(
    " UNION ALL ".join(f"SELECT '{t}' AS table_name, COUNT(*) AS row_count FROM {t}" for t in
                       ("network", "station", "station_epoch", "channel", "channel_epoch",
                        "source_file", "trace_segment", "sample")), conn)
print("All integrity, count and round-trip checks passed.\n")
display(table_rows)
display(constraint_checks)
pd.read_sql("SELECT file_name, gap_start_utc, gap_seconds FROM v_gap ORDER BY gap_start_us", conn)

## 7. Visualization
The helicorders are built from **samples read out of the database**, not from the miniSEED files. Each day is
placed on a regular 100 Hz grid (gaps stay blank), high-pass filtered one segment at a time so the filter never
runs across a gap, and drawn as 24 one-hour lines. To keep plotting fast, each line is reduced to its
min/max envelope per screen column, which looks the same as drawing every sample.

All three days share one amplitude scale, so they can be compared directly.

In [ ]:
@dataclass
class DayRecord:
    file_id: int
    file_name: str
    day_start: datetime
    sample_rate: float
    values: np.ndarray   # filtered, float32, NaN where there is no data
    raw_min: int
    raw_max: int
    raw_mean: float
    samples: int
    segments: int


def load_day_from_db(conn, file_id: int) -> DayRecord:
    segs = conn.execute(
        """SELECT segment_id, start_time_us, npts, sample_rate_hz, file_name
           FROM v_segment WHERE file_id = ? ORDER BY start_time_us""", (file_id,)
    ).fetchall()
    sr = segs[0][3]
    day_start = from_us(segs[0][1]).replace(hour=0, minute=0, second=0, microsecond=0)
    grid = np.full(int(round(86_400 * sr)), np.nan, dtype=np.float32)
    sos = signal.butter(4, HELI_HIGHPASS_HZ, btype="highpass", fs=sr, output="sos")
    min_len = 3 * (2 * len(sos) + 1) + 1  # shortest segment sosfiltfilt can pad
    raw_min, raw_max, raw_sum, total = np.inf, -np.inf, 0.0, 0
    for segment_id, start_us, npts, _, _ in segs:
        counts = fetch_segment_counts(conn, segment_id, npts)
        raw_min, raw_max = min(raw_min, counts.min()), max(raw_max, counts.max())
        raw_sum, total = raw_sum + counts.sum(), total + npts
        x = counts.astype(np.float64) - counts.mean()
        if npts >= min_len:
            x = signal.sosfiltfilt(sos, x)
        i0 = int(round((start_us - to_us(day_start)) * sr / 1e6))
        lo, hi = max(i0, 0), min(i0 + npts, grid.size)  # files run 10 ms into the next day
        if hi > lo:
            grid[lo:hi] = x[lo - i0:hi - i0]
    return DayRecord(file_id, segs[0][4], day_start, sr, grid, int(raw_min), int(raw_max),
                     raw_sum / total, total, len(segs))


days = [load_day_from_db(conn, fid) for (fid,) in conn.execute("SELECT file_id FROM source_file ORDER BY file_id")]

# One shared scale: the median over the days of each day's 99th-percentile absolute amplitude
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    heli_scale = float(np.median([np.nanpercentile(np.abs(d.values), 99) for d in days]))
print(f"Loaded {len(days)} days from the database; helicorder scale: {heli_scale:.1f} counts = {HELI_P99_HEIGHT} line spacings")


def plot_helicorder(ax, day: DayRecord, scale: float, show_utc_labels: bool, show_local_labels: bool,
                    bins_per_line: int = 1500) -> None:
    n_lines = 24 * 60 // HELI_LINE_MINUTES
    per_line = int(HELI_LINE_MINUTES * 60 * day.sample_rate)
    bin_size = max(1, per_line // bins_per_line)
    n_bins = per_line // bin_size
    lines = day.values[: n_lines * per_line].reshape(n_lines, per_line)[:, : n_bins * bin_size]
    binned = lines.reshape(n_lines, n_bins, bin_size)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)  # all-NaN bins are gaps
        lo, hi = np.nanmin(binned, axis=2), np.nanmax(binned, axis=2)
    minutes = (np.arange(n_bins) + 0.5) * bin_size / day.sample_rate / 60
    x = np.repeat(minutes, 2)
    colors = ("#111111", "#b22222", "#1f4e9c", "#1e7b3c")  # classic drum-recorder colour cycle
    for i in range(n_lines):
        y = np.column_stack([lo[i], hi[i]]).ravel() / scale * HELI_P99_HEIGHT
        y = np.clip(y, -HELI_CLIP, HELI_CLIP)
        ax.plot(x, i - y, lw=0.45, color=colors[i % len(colors)], rasterized=True)
    ax.set_xlim(0, HELI_LINE_MINUTES)
    ax.set_ylim(n_lines - 0.2, -0.8)
    ax.set_xticks(np.arange(0, HELI_LINE_MINUTES + 1, 10))
    ax.set_xlabel("Minutes past the hour")
    ax.set_yticks(range(n_lines))
    ax.set_yticklabels([f"{h * HELI_LINE_MINUTES // 60:02d}:00" for h in range(n_lines)] if show_utc_labels else [])
    if show_utc_labels:
        ax.set_ylabel("Hour (UTC)")
    if show_local_labels:
        twin = ax.twinx()
        twin.set_ylim(ax.get_ylim())
        twin.set_yticks(range(n_lines))
        twin.set_yticklabels([f"{(h + LOCAL_UTC_OFFSET_H) % 24:02d}:00" for h in range(n_lines)], fontsize=8)
        twin.set_ylabel(f"Hour ({LOCAL_TZ_NAME}, UTC{LOCAL_UTC_OFFSET_H:+d})")
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="x", color="0.85", lw=0.6)
    coverage = 100 * day.samples / (86_400 * day.sample_rate)
    ax.set_title(f"{day.day_start:%a %d %b %Y} (UTC)\n{day.file_name} · {coverage:.2f}% coverage",
                 fontsize=11)

In [ ]:
def station_epoch_at(conn, network: str, station: str, when_us: int):
    """The epoch in force at `when_us`, or the most recent one if the station wasn't operating then."""
    return conn.execute(
        """SELECT latitude, longitude, elevation_m, site_name, start_date, end_date,
                  (start_time_us <= :t AND (end_time_us IS NULL OR end_time_us > :t)) AS operating
           FROM v_station_epoch WHERE network = :net AND station = :sta
           ORDER BY operating DESC, start_time_us DESC LIMIT 1""",
        {"t": when_us, "net": network, "sta": station},
    ).fetchone()


mid_window_us = (to_us(window_start) + to_us(window_end)) // 2
station_rows = []
listed = set()
for role, df in (("data", channel_meta), ("interest", interest_meta), ("nearby", nearby_meta)):
    if df.empty:
        continue
    for net, sta in df[["network", "station"]].drop_duplicates().itertuples(index=False):
        if (net, sta) in listed:
            continue
        listed.add((net, sta))
        lat, lon, elev, site, start_date, end_date, operating = station_epoch_at(conn, net, sta, mid_window_us)
        station_rows.append(dict(role=role, network=net, station=sta, latitude=lat, longitude=lon,
                                 elevation_m=elev, site_name=site, operating=bool(operating),
                                 epoch=f"{start_date} → {end_date or 'open'}"))
map_stations = pd.DataFrame(station_rows)
map_stations

In [ ]:
MSH_SUMMIT = (46.1914, -122.1956)  # Mount St. Helens summit (lat, lon), for orientation only
ESRI_TOPO_URL = "https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}"
ROLE_STYLE = {
    "data": dict(color="#d7191c", size=260, label="Station in the miniSEED files"),
    "interest": dict(color="#2c7bb6", size=200, label="Station of interest (IRIS)"),
    "nearby": dict(color="#8c8c8c", size=70, label=f"Other station within {NEARBY_RADIUS_DEG}° (IRIS)"),
}


def map_extent(df: pd.DataFrame, pad_frac: float = 0.15, min_pad_deg: float = 0.02):
    lats = np.append(df.latitude.to_numpy(), MSH_SUMMIT[0])
    lons = np.append(df.longitude.to_numpy(), MSH_SUMMIT[1])
    pad_lat = max((lats.max() - lats.min()) * pad_frac, min_pad_deg)
    pad_lon = max((lons.max() - lons.min()) * pad_frac, min_pad_deg / math.cos(math.radians(lats.mean())))
    return lons.min() - pad_lon, lons.max() + pad_lon, lats.min() - pad_lat, lats.max() + pad_lat


def tiles_reachable() -> bool:
    try:
        return requests.get(ESRI_TOPO_URL.format(z=10, y=360, x=163), timeout=5).ok
    except requests.RequestException:
        return False


def add_map_axes(fig, subplot_spec):
    """Cartopy axes with an Esri topographic basemap if possible; plain lat/lon axes otherwise."""
    try:
        import cartopy.crs as ccrs
        import cartopy.io.img_tiles as cimgt
    except ImportError:
        ax = fig.add_subplot(subplot_spec)
        return ax, None, None

    class EsriTopo(cimgt.GoogleWTS):
        def _image_url(self, tile):
            x, y, z = tile
            return ESRI_TOPO_URL.format(z=z, y=y, x=x)

    tiles = EsriTopo(cache=True) if tiles_reachable() else None
    proj = tiles.crs if tiles else ccrs.PlateCarree()
    return fig.add_subplot(subplot_spec, projection=proj), tiles, ccrs.PlateCarree()


def plot_station_map(ax, tiles, data_crs, df: pd.DataFrame) -> None:
    lon0, lon1, lat0, lat1 = map_extent(df)
    kw = {"transform": data_crs} if data_crs else {}
    if data_crs:
        ax.set_extent([lon0, lon1, lat0, lat1], crs=data_crs)
        if tiles:
            zoom = int(np.clip(round(math.log2(360 / (lon1 - lon0))) + 1, 6, 14))
            ax.add_image(tiles, zoom)
            ax.text(0.99, 0.01, "Basemap: Esri World Topographic Map", transform=ax.transAxes,
                    ha="right", va="bottom", fontsize=6, color="0.3")
        gl = ax.gridlines(draw_labels=True, lw=0.4, color="0.5", alpha=0.6)
        gl.top_labels = gl.right_labels = False
        gl.xlabel_style = gl.ylabel_style = {"size": 7}
    else:
        ax.set_xlim(lon0, lon1)
        ax.set_ylim(lat0, lat1)
        ax.set_aspect(1 / math.cos(math.radians((lat0 + lat1) / 2)))
        ax.set_xlabel("Longitude (°)")
        ax.set_ylabel("Latitude (°)")
        ax.grid(color="0.85", lw=0.5)

    halo = [pe.withStroke(linewidth=3, foreground="white")]
    for role in ("nearby", "interest", "data"):  # draw the important ones on top
        sub = df[df.role == role]
        style = ROLE_STYLE[role]
        ax.scatter(sub.longitude, sub.latitude, marker="^", s=style["size"], c=style["color"],
                   edgecolors="black", linewidths=0.8, zorder=5, **kw)
        for r in sub.itertuples():
            label = f"{r.network}.{r.station}" + ("" if r.operating else "\n(not operating)")
            ax.text(r.longitude, r.latitude, f"  {label}", fontsize=9 if role != "nearby" else 6.5,
                    fontweight="bold" if role != "nearby" else "normal", va="center", zorder=6,
                    path_effects=halo, **kw)
    ax.scatter(MSH_SUMMIT[1], MSH_SUMMIT[0], marker="*", s=180, c="gold", edgecolors="black", zorder=4, **kw)

    # Scale bar in the lower-left corner, about a quarter of the map width
    km_per_deg_lon = 111.32 * math.cos(math.radians((lat0 + lat1) / 2))
    bar_km = max((k for k in (1, 2, 5, 10, 20, 50, 100) if k <= 0.25 * (lon1 - lon0) * km_per_deg_lon), default=1)
    bar_deg = bar_km / km_per_deg_lon
    bx, by = lon0 + 0.05 * (lon1 - lon0), lat0 + 0.06 * (lat1 - lat0)
    ax.plot([bx, bx + bar_deg], [by, by], color="black", lw=3, zorder=7, solid_capstyle="butt", **kw)
    ax.text(bx + bar_deg / 2, by + 0.02 * (lat1 - lat0), f"{bar_km} km", ha="center", fontsize=8,
            zorder=7, path_effects=halo, **kw)

    handles = [Line2D([], [], marker="^", ls="", markersize=9 if r != "nearby" else 6,
                      markerfacecolor=ROLE_STYLE[r]["color"], markeredgecolor="black", label=ROLE_STYLE[r]["label"])
               for r in ("data", "interest", "nearby") if (df.role == r).any()]
    handles.append(Line2D([], [], marker="*", ls="", markersize=11, markerfacecolor="gold",
                          markeredgecolor="black", label="Mount St. Helens summit"))
    ax.legend(handles=handles, loc="upper right", fontsize=7, framealpha=0.9)
    ax.set_title("Station locations (IRIS FDSN station service)", fontsize=11)

In [ ]:
def metadata_text(conn, days: list[DayRecord], stations: pd.DataFrame) -> str:
    """Metadata panel, built from database queries."""
    f = conn.execute("SELECT encoding, record_length, byte_order FROM source_file LIMIT 1").fetchone()
    seg = conn.execute("SELECT seed_id, sample_rate_hz, data_quality FROM v_segment LIMIT 1").fetchone()
    ch = conn.execute(
        """SELECT sensor_description, scale, scale_freq_hz, scale_units, sample_rate_hz,
                  strftime('%Y-%m-%d', start_time_us / 1e6, 'unixepoch')
           FROM channel_epoch ORDER BY start_time_us DESC LIMIT 1""").fetchone()
    files = {r[0]: r[1:] for r in conn.execute("SELECT file_id, size_bytes, record_count FROM source_file")}
    gaps = dict(conn.execute(
        "SELECT s.file_id, COUNT(*) FROM v_gap g JOIN source_file s ON s.file_name = g.file_name GROUP BY s.file_id"))
    primary = stations[stations.role == "data"].iloc[0]
    hemi = lambda v, pos, neg: f"{abs(v):.5f}°{pos if v >= 0 else neg}"
    seed = seg[0].replace("..", ".--.")  # show the blank location code as '--'

    lines = [
        "MINISEED",
        f"  Stream ID      {seed}",
        f"  Sample rate    {seg[1]:g} Hz",
        f"  Encoding       {f[0]}, {f[1]}-byte records, {'big' if f[2] == '>' else 'little'}-endian",
        f"  Data quality   {seg[2]}",
        "",
        "STATION (IRIS)",
        f"  Site           {primary.site_name}",
        f"  Position       {hemi(primary.latitude, 'N', 'S')}, {hemi(primary.longitude, 'E', 'W')}",
        f"  Elevation      {primary.elevation_m:,.0f} m",
    ]
    if ch:
        lines += [f"  Sensor         {ch[0] or 'n/a'}",
                  (f"  Sensitivity    {ch[1]:.4g} counts per {ch[3] or '?'} at {ch[2]:g} Hz"
                   if ch[1] is not None and ch[2] is not None else "  Sensitivity    n/a"),
                  f"  Channel epoch  since {ch[5]}"]
    lines += ["", "FILES (one UTC day each)"]
    for d in days:
        size, records = files[d.file_id]
        lines += [
            f"  {d.file_name}  {d.day_start:%Y-%m-%d}  {size / 1e6:.2f} MB, {records:,} records",
            f"    {d.samples:,} samples, {d.segments} segments, {gaps.get(d.file_id, 0)} gaps,"
            f" {100 * d.samples / (86_400 * d.sample_rate):.2f}% coverage",
            f"    counts min/max/mean  {d.raw_min:,} / {d.raw_max:,} / {d.raw_mean:,.1f}",
        ]
    lines += ["", "OTHER STATIONS ON THE MAP"]
    for r in stations[stations.role != "data"].sort_values(["role", "station"]).itertuples():
        tag = "*" if r.role == "interest" else " "
        lines.append(f" {tag}{r.network}.{r.station:<5} {r.latitude:8.4f} {r.longitude:9.4f} "
                     f"{(r.site_name or '')[:24]}")
    lines.append("  (* = station of interest)")
    return "\n".join(lines)


panel_text = metadata_text(conn, days, map_stations)
print(panel_text)

In [ ]:
def build_figure(days: list[DayRecord], stations: pd.DataFrame, text: str):
    fig = plt.figure(figsize=(26, 16))
    gs = GridSpec(2, 1 + len(days), figure=fig, width_ratios=[1.35] + [1] * len(days),
                  height_ratios=[1, 1.15], left=0.03, right=0.965, top=0.885, bottom=0.05,
                  wspace=0.12, hspace=0.14)
    seed = stations[stations.role == "data"].iloc[0]
    fig.suptitle(f"Mount St. Helens crater station {seed.network}.{seed.station}: "
                 f"three days of continuous seismic data", fontsize=22, fontweight="bold", y=0.975)
    fig.text(0.5, 0.925,
             f"24-hour helicorders built from the SQL database ({HELI_HIGHPASS_HZ} Hz high-pass, shared amplitude scale)"
             f"  ·  waveforms: SEP01–03.mseed (CHIGHT-USGS/SeismicData)  ·  station metadata: IRIS FDSN station service",
             ha="center", fontsize=12, color="0.3")

    ax_text = fig.add_subplot(gs[0, 0])
    ax_text.axis("off")
    ax_text.set_title("Metadata", loc="left", fontsize=13, fontweight="bold")
    ax_text.text(0, 1, text, va="top", ha="left", family="monospace", fontsize=8.6, transform=ax_text.transAxes)

    ax_map, tiles, data_crs = add_map_axes(fig, gs[1, 0])
    plot_station_map(ax_map, tiles, data_crs, stations)

    for i, day in enumerate(days):
        ax = fig.add_subplot(gs[:, 1 + i])
        plot_helicorder(ax, day, heli_scale, show_utc_labels=(i == 0), show_local_labels=(i == len(days) - 1))
    return fig


fig = build_figure(days, map_stations, panel_text)
out_png = OUTPUT_DIR / "sep_seismic_overview.png"
fig.savefig(out_png, dpi=150, bbox_inches="tight")
print(f"Saved {out_png}")
plt.show()

### Optional: interactive map
Folium comes with Colab, so the same stations can be shown on a map you can pan and zoom. The static map above is
the required one; this is extra.

In [ ]:
try:
    import folium

    m = folium.Map(location=[map_stations.latitude.mean(), map_stations.longitude.mean()], zoom_start=11,
                   tiles="OpenTopoMap", attr="© OpenTopoMap (CC-BY-SA), © OpenStreetMap contributors")
    for r in map_stations.itertuples():
        folium.RegularPolygonMarker(
            location=[r.latitude, r.longitude], number_of_sides=3, rotation=30,
            radius=11 if r.role != "nearby" else 6, color="black", weight=1, fill=True, fill_opacity=0.9,
            fill_color=ROLE_STYLE[r.role]["color"],
            popup=folium.Popup(f"<b>{r.network}.{r.station}</b><br>{r.site_name}<br>"
                               f"{r.latitude:.5f}, {r.longitude:.5f}<br>{r.elevation_m:,.0f} m<br>epoch {r.epoch}",
                               max_width=260),
            tooltip=f"{r.network}.{r.station}",
        ).add_to(m)
    display(m)
except ImportError:
    print("folium is not installed; skipping the interactive map")

## Summary
* **Import:** three Steim-2 miniSEED files (`CC.SEP.--.EHZ`, 100 Hz, one UTC day each) uploaded
  before the run, checked against known SHA-256 hashes and read with ObsPy.
* **Database:** `outputs/seismic.db`, a SQLite schema in third normal form with keys, foreign keys,
  `UNIQUE`/`CHECK` constraints, a cross-table trigger and views for derived values. Every sample is stored, and the
  load is checked by row counts and an exact round-trip comparison.
* **Visualization:** `outputs/sep_seismic_overview.png`, with a title, a metadata panel generated from SQL,
  helicorders built from database samples, and a station map with SEP, the stations of interest HOA and SUG,
  and other stations operating nearby, all located through the IRIS FDSN station service.

To keep the outputs, download `outputs/` from the Colab file browser, or open the database with any SQLite client.

In [ ]:
conn.close()